# Quantum Phase Estimation

Estimate the eigenvalue phase of the **T gate** (θ = π/4). With 3 precision qubits, the resolution is 1/8, matching the T gate exactly.

In [ ]:
import pennylane as qml
import numpy as np

## QPE circuit

Apply controlled-U^(2^k) for each precision qubit k, then inverse QFT.

In [ ]:
N_PREC = 3
N_TOTAL = N_PREC + 1
dev = qml.device("default.qubit", wires=N_TOTAL)

def controlled_unitaries():
    for k in range(N_PREC):
        angle = (np.pi / 4) * (2**k)
        qml.ctrl(qml.RZ, control=k)(angle, wires=N_PREC)

@qml.qnode(dev)
def qpe_circuit():
    qml.PauliX(wires=N_PREC)
    qml.Hadamard(wires=range(N_PREC))
    controlled_unitaries()
    qml.adjoint(qml.QFT(wires=list(range(N_PREC))))
    return qml.probs(wires=range(N_PREC))

print("Circuit:")
print(qml.draw(qpe_circuit)())

## Results

In [ ]:
probs = qpe_circuit()
measured = np.argmax(probs)
phase_measured = measured / (2**N_PREC)
phase_actual = 1 / 8

print("Measurement probabilities:")
for i, p in enumerate(probs):
    if p > 0.01:
        phase_est = i / (2**N_PREC)
        print(f"  |{i:0{N_PREC}b}\u27e9  p={p:.4f}  phase={phase_est:.4f}\u00d72\u03c0")

print(f"\nMeasured: |{measured:0{N_PREC}b}\u27e9")
print(f"Phase estimate: {phase_measured:.4f} \u00d7 2\u03c0 = {phase_measured * 2 * np.pi:.4f}")
print(f"Expected:       {phase_actual:.4f} \u00d7 2\u03c0 = {phase_actual * 2 * np.pi:.4f}")
print(f"Exact match: {np.isclose(phase_measured, phase_actual)}")